# NICTO From-Scratch Super Training
Trains SuperNeuralCore from random initialization on Kaggle T4/P100 GPU
No pre-trained weights — pure from-scratch learning using 361K ChatML entries across 57 domains

## 1. Environment Setup

In [ ]:
import os, sys, json, time, math, gc, glob, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import Counter
from dataclasses import dataclass

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({props.total_memory / 1e9:.1f}GB VRAM)")

In [ ]:
# Clone the NICTO repo
!git clone https://github.com/stephenwahogo0-arch/NICTO.git /kaggle/working/nicto_repo
sys.path.insert(0, '/kaggle/working/nicto_repo')
print('Repo cloned, path set')

In [ ]:
# Minimal dependencies — we only need transformers for the tokenizer
!pip install -q transformers datasets sentencepiece tokenizers

## 2. Tokenizer Setup
We use the GPT-2 tokenizer (50257 vocab) with ChatML special tokens added.

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Add ChatML special tokens
chatml_tokens = ['<|im_start|>', '<|im_end|>', '<|im_sep|>', '<|system|>', '<|user|>', '<|assistant|>']
tokenizer.add_special_tokens({
    'pad_token': '<|pad|>',
    'eos_token': '<|im_end|>',
    'bos_token': '<|im_start|>',
    'additional_special_tokens': chatml_tokens
})

VOCAB_SIZE = len(tokenizer)
print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'PAD id: {tokenizer.pad_token_id}, EOS id: {tokenizer.eos_token_id}')

## 3. Load Training Data
Data is mounted from the Kaggle dataset `stephenwahogo/nicto-super-training-data-v3`

In [ ]:
DATA_DIR = '/kaggle/input/nicto-super-training-data-v3'
data_path = f'{DATA_DIR}/super_v3_chatml.jsonl'

print(f'Data path: {data_path}')
!ls -lh {DATA_DIR}/
!wc -l {data_path}

# Load a sample to inspect
with open(data_path, 'r') as f:
    sample = json.loads(f.readline())
print(f'\nSample keys: {list(sample.keys())}')
print(f'Messages: {len(sample.get("messages", []))}')

In [ ]:
# Load all entries
entries = []
with open(data_path, 'r') as f:
    for line in f:
        entries.append(json.loads(line))
print(f'Loaded {len(entries)} training entries')

# Analyze categories
cats = Counter()
for e in entries:
    m = e.get('metadata', {})
    if isinstance(m, dict):
        cats[m.get('domain', m.get('category', 'unknown'))] += 1
print(f'Domains: {len(cats)}')
for dom, cnt in cats.most_common(15):
    print(f'  {dom}: {cnt}')

## 4. Tokenize Dataset

In [ ]:
def format_chatml(messages):
    """Convert messages list to ChatML text."""
    parts = []
    for m in messages:
        role = m.get('role', 'user')
        content = m.get('content', '')
        if role == 'system':
            parts.append(f'<|im_start|>system\n{content}<|im_end|>')
        elif role == 'user':
            parts.append(f'<|im_start|>user\n{content}<|im_end|>')
        elif role == 'assistant':
            parts.append(f'<|im_start|>assistant\n{content}<|im_end|>')
    return '\n'.join(parts)

MAX_SEQ_LEN = 1024

def tokenize_entry(e):
    """Tokenize one entry. Returns input_ids and labels (same, with loss masked on user text)."""
    text = format_chatml(e.get('messages', []))
    tokens = tokenizer.encode(text, truncation=True, max_length=MAX_SEQ_LEN)
    return tokens

# Tokenize first 50000 entries for initial training (fits in memory)
TRAIN_SUBSET = min(50000, len(entries))
print(f'Tokenizing {TRAIN_SUBSET} entries...')
all_tokens = []
t0 = time.time()
for i, e in enumerate(entries[:TRAIN_SUBSET]):
    tokens = tokenize_entry(e)
    if len(tokens) >= 10:
        all_tokens.append(tokens)
    if (i + 1) % 10000 == 0:
        elapsed = time.time() - t0
        print(f'  {i+1}/{TRAIN_SUBSET} ({elapsed:.1f}s)')

print(f'Tokenized {len(all_tokens)} sequences')
print(f'Avg length: {np.mean([len(t) for t in all_tokens]):.1f} tokens')

## 5. SuperNeuralCore Initialization
Initialize from scratch with Small config (327M params)

In [ ]:
from nicto_neural.neural.super_config import SuperConfig, CONFIG_MAP
from nicto_neural.neural.super_core import SuperNeuralCore

# Small config adapted for GPT-2 tokenizer
config = SuperConfig(
    d_model=768,
    n_heads=12,
    n_kv_heads=4,
    n_layers=12,
    d_ff=2048,
    n_experts=4,
    n_active_experts=1,
    max_seq_len=MAX_SEQ_LEN,
    n_brain_heads=9,
    vocab_size=VOCAB_SIZE,
    dropout=0.1,
    use_rope=True,
    use_flash_attn=True,
)

# Estimate parameters
est = config.estimate_params()
print(f"Estimated params: {est['total_billions']:.3f}B ({est['total']:,})")

# Initialize from scratch — random weights
print('Initializing SuperNeuralCore from scratch...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SuperNeuralCore(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Actual params: {total_params:,} ({total_params/1e9:.3f}B)')
print(f'Trainable: {trainable_params:,}')
print(f'Device: {device}')

## 6. Training Setup
SFT training on ChatML next-token prediction

In [ ]:
BATCH_SIZE = 4
GRAD_ACCUM = 8
EPOCHS = 3
LEARNING_RATE = 3e-4
WARMUP_STEPS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=(len(all_tokens) * EPOCHS) // (BATCH_SIZE * GRAD_ACCUM), eta_min=1e-5
)

print(f'Batch size: {BATCH_SIZE}')
print(f'Grad accum: {GRAD_ACCUM}')
print(f'Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'Epochs: {EPOCHS}')
print(f'Steps per epoch: {len(all_tokens) // (BATCH_SIZE * GRAD_ACCUM)}')
print(f'Total steps: {len(all_tokens) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)}')
print(f'Training from scratch with {len(all_tokens)} sequences')

In [ ]:
def collate_batch(tokens_list, max_len):
    """Pad sequences to same length."""
    batch_size = len(tokens_list)
    input_ids = torch.full((batch_size, max_len), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)
    
    for i, tokens in enumerate(tokens_list):
        seq_len = min(len(tokens), max_len)
        input_ids[i, :seq_len] = torch.tensor(tokens[:seq_len], dtype=torch.long)
        labels[i, :seq_len] = torch.tensor(tokens[:seq_len], dtype=torch.long)
        attention_mask[i, :seq_len] = 1
    
    return input_ids, labels, attention_mask


def get_batch(tokens_list, batch_size, max_len, shuffle=True):
    """Yield batches from token list."""
    indices = list(range(len(tokens_list)))
    if shuffle:
        random.shuffle(indices)
    for i in range(0, len(indices), batch_size):
        batch_indices = indices[i:i+batch_size]
        batch_tokens = [tokens_list[j] for j in batch_indices]
        yield collate_batch(batch_tokens, max_len)

## 7. Training Loop — Epoch 1
Loss should decrease from ~9.0 to ~4.0 over the first epoch

In [ ]:
training_log = []
global_step = 0
total_steps = (len(all_tokens) * EPOCHS) // (BATCH_SIZE * GRAD_ACCUM)

for epoch in range(1, EPOCHS + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'{"="*60}')
    
    model.train()
    epoch_losses = []
    epoch_accs = []
    optimizer.zero_grad()
    
    for batch_idx, (input_ids, labels, attention_mask) in enumerate(
        get_batch(all_tokens, BATCH_SIZE, MAX_SEQ_LEN)
    ):
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        
        output = model(input_ids)
        logits = output['logits']
        
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
        )
        
        loss = loss / GRAD_ACCUM
        loss.backward()
        
        if (batch_idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            
            with torch.no_grad():
                acc = (shift_logits.argmax(dim=-1) == shift_labels).float().mean().item()
                ppl = math.exp(min(loss.item() * GRAD_ACCUM, 20))
                
            epoch_losses.append(loss.item() * GRAD_ACCUM)
            epoch_accs.append(acc)
            
            if global_step % 20 == 0:
                lr_now = scheduler.get_last_lr()[0]
                avg_loss = np.mean(epoch_losses[-50:]) if len(epoch_losses) >= 50 else np.mean(epoch_losses)
                avg_acc = np.mean(epoch_accs[-50:]) if len(epoch_accs) >= 50 else np.mean(epoch_accs)
                print(f'  Step {global_step:>5}/{total_steps} | loss: {loss.item() * GRAD_ACCUM:.4f} | ' +
                      f'avg_loss: {avg_loss:.4f} | acc: {avg_acc:.4f} | ppl: {ppl:.2f} | lr: {lr_now:.2e}')
            
            training_log.append({
                'step': global_step,
                'epoch': epoch,
                'loss': loss.item() * GRAD_ACCUM,
                'accuracy': acc,
                'perplexity': ppl,
                'lr': scheduler.get_last_lr()[0],
            })
    
    # End of epoch
    epoch_avg_loss = np.mean(epoch_losses) if epoch_losses else 100
    epoch_avg_acc = np.mean(epoch_accs) if epoch_accs else 0
    print(f'\nEpoch {epoch} complete: avg_loss={epoch_avg_loss:.4f}, avg_acc={epoch_avg_acc:.4f}')
    
    # Save checkpoint after each epoch
    ckpt_dir = f'/kaggle/working/checkpoints/epoch_{epoch}'
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'config': config,
        'training_log': training_log,
        'avg_loss': epoch_avg_loss,
        'avg_accuracy': epoch_avg_acc,
    }, f'{ckpt_dir}/model.pt')
    print(f'Checkpoint saved: {ckpt_dir}/model.pt ({os.path.getsize(ckpt_dir + "/model.pt") / 1e9:.2f}GB)')

## 8. Save Final Model

In [ ]:
# Save final model
final_dir = '/kaggle/working/final_model'
os.makedirs(final_dir, exist_ok=True)

final_ckpt = {
    'model_state_dict': model.state_dict(),
    'config': config,
    'training_log': training_log,
    'vocab_size': VOCAB_SIZE,
    'tokenizer_name': 'gpt2',
    'chatml_tokens': tokenizer.additional_special_tokens,
    'num_entries_trained': len(all_tokens),
    'total_steps': global_step,
    'final_avg_loss': np.mean([l['loss'] for l in training_log[-100:]]) if training_log else 0,
}
torch.save(final_ckpt, f'{final_dir}/super_neural_final.pt')

import json
with open(f'{final_dir}/training_report.json', 'w') as f:
    json.dump({
        'model_params': total_params,
        'trainable_params': trainable_params,
        'config': {
            'd_model': config.d_model,
            'n_heads': config.n_heads,
            'n_kv_heads': config.n_kv_heads,
            'n_layers': config.n_layers,
            'd_ff': config.d_ff,
            'n_experts': config.n_experts,
            'n_active_experts': config.n_active_experts,
            'vocab_size': config.vocab_size,
            'max_seq_len': config.max_seq_len,
            'total_params': total_params,
        },
        'training': {
            'entries': len(all_tokens),
            'epochs': EPOCHS,
            'total_steps': global_step,
            'batch_size': BATCH_SIZE,
            'grad_accum': GRAD_ACCUM,
            'effective_batch': BATCH_SIZE * GRAD_ACCUM,
            'learning_rate': LEARNING_RATE,
        },
        'final_metrics': {
            'avg_loss_final': final_ckpt['final_avg_loss'],
            'perplexity_final': math.exp(min(final_ckpt['final_avg_loss'], 20)),
        },
        'timestamp': time.time(),
    }, f, indent=2)

!ls -lh {final_dir}/
print(f'\nFinal model saved to {final_dir}/super_neural_final.pt')

## 9. Validation — Test Generation

In [ ]:
def generate_text(prompt, max_new=100, temperature=0.8, top_k=40):
    """Generate text from prompt."""
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        for _ in range(max_new):
            output = model(input_ids)
            logits = output['logits'][:, -1, :] / temperature
            
            if top_k > 0:
                top_k_vals, _ = torch.topk(logits, top_k, dim=-1)
                logits[logits < top_k_vals[:, -1:]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
    
    return tokenizer.decode(input_ids[0], skip_special_tokens=False)

# Test prompts across domains
test_prompts = [
    '<|im_start|>user\nWhat is a neural network?<|im_end|>\n<|im_start|>assistant\n',
    '<|im_start|>user\nWrite a Python function to reverse a string<|im_end|>\n<|im_start|>assistant\n',
    '<|im_start|>user\nExplain SQL injection<|im_end|>\n<|im_start|>assistant\n',
    '<|im_start|>user\nWhat is the capital of France?<|im_end|>\n<|im_start|>assistant\n',
]

print('\\n--- GENERATION TESTS ---')
for prompt in test_prompts:
    output = generate_text(prompt, max_new=80)
    print(f'\\nPrompt: {prompt}')
    print(f'Output: {output}')

In [ ]:
# Calculate validation loss on a separate subset
VAL_SUBSET = min(1000, len(entries) - TRAIN_SUBSET)
if VAL_SUBSET > 0:
    val_start = TRAIN_SUBSET
    val_tokens = []
    for e in entries[val_start:val_start + VAL_SUBSET]:
        tokens = tokenize_entry(e)
        if len(tokens) >= 10:
            val_tokens.append(tokens)
    
    model.eval()
    val_losses = []
    with torch.no_grad():
        for input_ids, labels, _ in get_batch(val_tokens, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False):
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            output = model(input_ids)
            logits = output['logits']
            
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
            val_losses.append(loss.item())
    
    val_loss = np.mean(val_losses)
    val_ppl = math.exp(min(val_loss, 20))
    print(f'\\nValidation: {len(val_tokens)} sequences')
    print(f'Val loss: {val_loss:.4f}, Val perplexity: {val_ppl:.2f}')
else:
    print('No validation data available (all entries used for training)')

## 10. Compress Results for Download

In [ ]:
# Package everything
!mkdir -p /kaggle/working/output
!cp -r /kaggle/working/final_model /kaggle/working/output/
!cp -r /kaggle/working/checkpoints /kaggle/working/output/

# Save training log as JSON
with open('/kaggle/working/output/training_log.json', 'w') as f:
    json.dump(training_log, f)

# Copy config
!cp /kaggle/working/nicto_repo/nicto_neural/neural/super_config.py /kaggle/working/output/
!cp /kaggle/working/nicto_repo/nicto_neural/neural/super_core.py /kaggle/working/output/
!cp /kaggle/working/nicto_repo/nicto_neural/neural/heads.py /kaggle/working/output/
!cp /kaggle/working/nicto_repo/nicto_neural/neural/reasoning.py /kaggle/working/output/

!zip -r /kaggle/working/nicto_super_neural_v1.zip /kaggle/working/output/
!ls -lh /kaggle/working/nicto_super_neural_v1.zip
print('\\nTraining complete. Download /kaggle/working/nicto_super_neural_v1.zip')

In [ ]:
print('=' * 60)
print('NICTO FROM-SCRATCH TRAINING SUMMARY')
print('=' * 60)
print(f'Model: SuperNeuralCore Small ({total_params:,} params)')
print(f'Config: d={config.d_model}, L={config.n_layers}, H={config.n_heads}, ' +
      f'KV={config.n_kv_heads}, ff={config.d_ff}, experts={config.n_experts}')
print(f'Vocab: {VOCAB_SIZE} tokens (GPT-2 + ChatML)')
print(f'Training data: {len(all_tokens)} sequences')
print(f'Epochs: {EPOCHS}')
print(f'Total steps: {global_step}')
print(f'Final avg loss: {final_ckpt["final_avg_loss"]:.4f}')
print(f'Final perplexity: {math.exp(min(final_ckpt["final_avg_loss"], 20)):.2f}')
print(f'Brain heads: 9 (analytical, creative, knowledge, strategic, ' +
      'intuitive, ethical, linguistic, temporal, retrieval)')
print(f'Reasoning paths: 6 (deductive, inductive, abductive, analogical, critical, creative)')
print(f'Training trained from: Random initialization (no pre-trained weights)')
print(f'=' * 60)